In [13]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import jax
import diffrax
from diffrax import ODETerm, SaveAt, diffeqsolve
import jax.numpy as jnp
import numpy as np
from jax.scipy.optimize import minimize
from triangular_transport.mcmc.adaptive_mcmc import AdaptiveMCMC
from triangular_transport.mcmc.mala import MALA
# from adap_mcmc import AdaptiveMCMC
from triangular_transport.flows.dataloaders import log_normal_reference_sampler
from lv import LV

In [3]:
T = 18
seed = np.random.choice(100000)
no_samples = 100000
u_true = jnp.array([0.83194674, 0.04134147, 1.0823151, 0.03991483]) # TODO: Need to figure out how to get u_true here from a y_obs.
u_moderate = np.array([1.8026501 , 0.05537019, 0.54760194, 0.04671995])
u_rare = np.array([0.5807536 , 0.06636621, 0.32894754, 0.09824311])
# sigma = jnp.sqrt(0.1).item()
sigma = jnp.sqrt(0.9).item()
lv = LV(
    seed=0,
    no_samples=no_samples,
    prior_sampler=log_normal_reference_sampler,
    likelihood_sampler=log_normal_reference_sampler,
    normalize=False,
    u_true=u_true,
    sigma=sigma,
    dt0=0.1,
)

In [26]:
term = ODETerm(lv.lv_ode)
args = u_moderate
t0 = 0
t1 = 20
ts = jnp.arange(t0, t1, step=2) # This takes care of the 1: indexing.
# ts = jnp.linspace(0, t1, 9)
saveat = SaveAt(ts=ts)
sol = diffeqsolve(
    term,
    lv.solver,
    t0,
    t1,
    lv.dt0,
    lv.y0_start,
    args=args,
    saveat=saveat,
)
ys = sol.ys

In [27]:
ys

Array([[3.0000000e+01, 1.0000000e+00],
       [6.2497950e-01, 1.5079979e+02],
       [8.6173828e-04, 5.0695946e+01],
       [1.0458858e-03, 1.6957464e+01],
       [1.2291689e-02, 5.6740012e+00],
       [3.0851674e-01, 1.9137089e+00],
       [9.9151344e+00, 8.2740748e-01],
       [6.4161545e+01, 1.4074344e+02],
       [2.2924407e-03, 7.1616493e+01],
       [6.8067759e-04, 2.3955622e+01]], dtype=float32)

In [28]:
lv.solve_lv(lv.y0_start, u_moderate)

Array([[6.2497950e-01, 1.5079979e+02],
       [8.6173828e-04, 5.0695946e+01],
       [1.0458858e-03, 1.6957464e+01],
       [1.2291689e-02, 5.6740012e+00],
       [3.0851674e-01, 1.9137089e+00],
       [9.9151344e+00, 8.2740748e-01],
       [6.4161545e+01, 1.4074344e+02],
       [2.2924407e-03, 7.1616493e+01],
       [6.8067759e-04, 2.3955622e+01]], dtype=float32)